In [ ]:
import math

from origami_jsynth.baselines._preprocessing import records_to_dataframe
from origami_jsynth.data import load_jsonl
from origami_jsynth.registry import get_dataset

DATASET = "yelp"  # change this to run on a different dataset

info = get_dataset(DATASET)


def df_to_records(df):
    """Convert DataFrame to list of dicts, dropping NaN keys (= missing values)."""
    records = df.to_dict(orient="records")
    return [{k: v for k, v in r.items() if not (isinstance(v, float) and math.isnan(v))} for r in records]


# Make sure the data files exist, by running `origami-jsynth data --dataset <name>` in the terminal.
train_json = load_jsonl(f"../results/{DATASET}/data/train.jsonl")
test_json = load_jsonl(f"../results/{DATASET}/data/test.jsonl")

print(f"Number of training examples: {len(train_json)}")
print(f"Number of test examples: {len(test_json)}")

# Flatten + type-separate for the flat model (same preprocessing baselines use).
# Exclude target column so it stays as a single field.
exclude = [info.target_column] if info.target_column else []
train_df, _ = records_to_dataframe(train_json, tabular=info.tabular, exclude_columns=exclude)
test_df, _ = records_to_dataframe(test_json, tabular=info.tabular, exclude_columns=exclude)

train_flat = df_to_records(train_df)
test_flat = df_to_records(test_df)

print(f"\nFlat training examples: {len(train_flat)}")
print(f"Flat columns: {len(train_df.columns)}")
print(f"Sample keys (first 10): {sorted(train_flat[0].keys())[:10]}...")

In [ ]:
import yaml
from origami import (
    DataConfig,
    InferenceConfig,
    ModelConfig,
    OrigamiConfig,
    OrigamiPipeline,
    TrainingConfig,
)

# Load best config from yaml
with open(f"../configs/{DATASET}.yaml") as f:
    config_yaml = yaml.safe_load(f)


config = OrigamiConfig(
    data=DataConfig(**config_yaml["data"]),
    model=ModelConfig(**config_yaml["model"]),
    training=TrainingConfig(**config_yaml["training"]),
    inference=InferenceConfig(**config_yaml["inference"]),
)

# num samples to generate
n_samples = len(train_json)

In [ ]:
from origami.training import NotebookCallback

from origami_jsynth.evaluation.evaluate import evaluate_synthetic_data

task_type = "classification" if info.task_type in ("binclass", "multiclass") else info.task_type

pipeline_flat = OrigamiPipeline(config)
pipeline_flat.fit(
    data=train_flat,
    eval_data=test_flat,
    callbacks=[NotebookCallback()],
)

synth_flat = pipeline_flat.generate(n_samples)
print(f"Flat model: generated {len(synth_flat)} samples")

eval_flat = evaluate_synthetic_data(
    train_records=train_flat,
    eval_records=test_flat,
    synthetic_records=synth_flat,
    target_column=info.target_column,
    task_type=task_type,
    privacy=False,
)
print("=== Flat model ===")
print(eval_flat)

In [ ]:
pipeline_json = OrigamiPipeline(config)
pipeline_json.fit(
    data=train_json,
    eval_data=test_json,
    callbacks=[NotebookCallback()],
)

synth_json = pipeline_json.generate(n_samples)
print(f"JSON model: generated {len(synth_json)} samples")

eval_json = evaluate_synthetic_data(
    train_records=train_json,
    eval_records=test_json,
    synthetic_records=synth_json,
    target_column=info.target_column,
    task_type=task_type,
    privacy=False,
)
print("=== JSON model ===")
print(eval_json)